# Phase 9: Rating Analysis
## Deep Dive into Restaurant Ratings
**Objective:** Analyze rating patterns, trends, and factors affecting ratings
**Output:** Dashboard-ready insights on rating distributions and correlations

In [19]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

print("✓ Libraries imported")

✓ Libraries imported


In [20]:
# Load cleaned data
df = pd.read_csv('Zomato_datasets/zomato_cleaned.csv')

# Create rating categories
def categorize_rating(rate):
    if rate >= 4.5:
        return 'Excellent (4.5+)'
    elif rate >= 4.0:
        return 'Very Good (4.0-4.5)'
    elif rate >= 3.5:
        return 'Good (3.5-4.0)'
    elif rate >= 3.0:
        return 'Average (3.0-3.5)'
    else:
        return 'Poor (<3.0)'

df['rating_category'] = df['rating'].apply(categorize_rating)

print(f"Dataset loaded: {df.shape}")
print(f"Rating Range: {df['rating'].min()} - {df['rating'].max()}")

Dataset loaded: (7105, 13)
Rating Range: 1.8 - 4.9


In [21]:
# 1. Rating Category Distribution
rating_dist = df['rating_category'].value_counts().sort_index()

colors = ['#e74c3c', '#f39c12', '#f1c40f', '#2ecc71', '#27ae60']

fig = px.pie(
    values=rating_dist.values,
    names=rating_dist.index,
    title='Restaurant Distribution by Rating Category',
    color_discrete_sequence=colors
)

fig.update_traces(textposition='inside', textinfo='percent+label')
fig.update_layout(height=600)
fig.show()

print("Rating Category Distribution:")
print(rating_dist)
print(f"\nPercentage Distribution:")
print((rating_dist / len(df) * 100).round(2))

Rating Category Distribution:
rating_category
Average (3.0-3.5)      1976
Excellent (4.5+)         98
Good (3.5-4.0)         2803
Poor (<3.0)            1076
Very Good (4.0-4.5)    1152
Name: count, dtype: int64

Percentage Distribution:
rating_category
Average (3.0-3.5)      27.81
Excellent (4.5+)        1.38
Good (3.5-4.0)         39.45
Poor (<3.0)            15.14
Very Good (4.0-4.5)    16.21
Name: count, dtype: float64


In [22]:
# 2. Rating by Restaurant Type
top_restaurant_types = df['restaurant_type'].value_counts().head(10).index
df_top = df[df['restaurant_type'].isin(top_restaurant_types)]

fig = px.box(
    df_top,
    x='restaurant_type',
    y='rating',
    title='Rating Distribution by Top 10 Restaurant Types',
    labels={'restaurant_type': 'Restaurant Type', 'rating': 'Rating'},
    color='restaurant_type',
    points='outliers'
)

fig.update_layout(height=600, xaxis_tickangle=-45, showlegend=False)
fig.show()

print(f"\nAverage Rating by Restaurant Type (Top 10):")
avg_rating_type = df_top.groupby('restaurant_type')['rating'].agg(['mean', 'count', 'std']).round(2)
print(avg_rating_type.sort_values('mean', ascending=False))


Average Rating by Restaurant Type (Top 10):
                    mean  count   std
restaurant_type                      
Casual Dining, Bar  3.87    123  0.51
Cafe                3.69    403  0.49
Bar                 3.68     82  0.43
Casual Dining       3.59   1634  0.48
Dessert Parlor      3.57    217  0.45
Bakery              3.48    154  0.40
Beverage Shop       3.41    118  0.40
Quick Bites         3.40   2840  0.40
Delivery            3.38    358  0.41
Takeaway, Delivery  3.29    289  0.39


In [23]:
# 3. Rating vs Number of Ratings Received
fig = px.scatter(
    df,
    x='num_ratings',
    y='rating',
    color='rating',
    size='avg_cost',
    hover_name='restaurant_name',
    title='Rating vs Number of Ratings Received',
    labels={'num_ratings': 'Number of Ratings', 'rating': 'Average Rating', 'avg_cost': 'Avg Cost'},
    color_continuous_scale='Viridis'
)

fig.update_layout(height=600)
fig.show()

# Calculate correlation
corr = df['rating'].corr(df['num_ratings'])
print(f"\nCorrelation between Rating and Number of Ratings: {corr:.3f}")


Correlation between Rating and Number of Ratings: 0.380


In [24]:
# 4. Impact of Online Order on Ratings
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Rating Distribution with Online Order', 'Rating Distribution without Online Order'),
    specs=[[{'type': 'histogram'}, {'type': 'histogram'}]]
)

with_order = df[df['has_online_order'] == 1]['rating']
without_order = df[df['has_online_order'] == 0]['rating']

fig.add_trace(go.Histogram(x=with_order, nbinsx=30, name='With Online Order', marker_color='green'), row=1, col=1)
fig.add_trace(go.Histogram(x=without_order, nbinsx=30, name='Without Online Order', marker_color='red'), row=1, col=2)

fig.update_xaxes(title_text='Rating', row=1, col=1)
fig.update_xaxes(title_text='Rating', row=1, col=2)
fig.update_yaxes(title_text='Count', row=1, col=1)
fig.update_yaxes(title_text='Count', row=1, col=2)

fig.update_layout(height=500, title_text='Impact of Online Order on Ratings', showlegend=False)
fig.show()

print(f"Average Rating with Online Order: {with_order.mean():.2f}")
print(f"Average Rating without Online Order: {without_order.mean():.2f}")
print(f"Difference: {with_order.mean() - without_order.mean():.2f}")

Average Rating with Online Order: 3.55
Average Rating without Online Order: 3.47
Difference: 0.08


In [25]:
# 5. Impact of Table Booking on Ratings
with_booking = df[df['has_table_booking'] == 1]['rating']
without_booking = df[df['has_table_booking'] == 0]['rating']

fig = go.Figure()

fig.add_trace(go.Box(y=with_booking, name='With Table Booking', marker_color='blue'))
fig.add_trace(go.Box(y=without_booking, name='Without Table Booking', marker_color='orange'))

fig.update_layout(
    title='Impact of Table Booking on Ratings',
    yaxis_title='Rating',
    height=500
)

fig.show()

print(f"Average Rating with Table Booking: {with_booking.mean():.2f}")
print(f"Average Rating without Table Booking: {without_booking.mean():.2f}")
print(f"Difference: {with_booking.mean() - without_booking.mean():.2f}")

Average Rating with Table Booking: 4.04
Average Rating without Table Booking: 3.45
Difference: 0.58


In [26]:
# 6. Top 15 Highest Rated Areas
area_stats = df.groupby('area').agg({
    'rating': ['mean', 'count', 'std']
}).reset_index()

area_stats.columns = ['area', 'avg_rating', 'restaurant_count', 'std_rating']
area_stats = area_stats[area_stats['restaurant_count'] >= 5]  # Filter areas with at least 5 restaurants
area_stats = area_stats.sort_values('avg_rating', ascending=False).head(15)

fig = px.bar(
    area_stats,
    x='avg_rating',
    y='area',
    color='avg_rating',
    title='Top 15 Highest Rated Areas (min 5 restaurants)',
    labels={'avg_rating': 'Average Rating', 'area': 'Area'},
    color_continuous_scale='RdYlGn'
)

fig.update_layout(height=600, showlegend=False)
fig.show()

print("\nTop 15 Highest Rated Areas:")
print(area_stats[['area', 'avg_rating', 'restaurant_count']])


Top 15 Highest Rated Areas:
                              area  avg_rating  restaurant_count
4                     Brigade Road    3.684267               464
20                    Lavelle Road    3.630496               141
22                    Malleshwaram    3.629602               402
7                    Church Street    3.606494                77
11                     Indiranagar    3.592308               455
6   Byresandra,Tavarekere,Madiwala    3.583960               798
0                     Banashankari    3.568802               359
21                         MG Road    3.540625                32
19           Koramangala 7th Block    3.530556                36
17           Koramangala 5th Block    3.528571                35
18           Koramangala 6th Block    3.525352                71
9                      Frazer Town    3.517460               126
16           Koramangala 4th Block    3.516049               162
2                     Basavanagudi    3.514019               

In [27]:
# 7. Statistical Summary
print("\n" + "="*70)
print("RATING ANALYSIS SUMMARY")
print("="*70)

print(f"\nDescriptive Statistics:")
print(df['rating'].describe().round(3))

print(f"\n\nSkewness: {stats.skew(df['rating']):.3f}")
print(f"Kurtosis: {stats.kurtosis(df['rating']):.3f}")

print(f"\n\nRating Distribution:")
print(df['rating_category'].value_counts().sort_index())

print("\n✓ Rating Analysis Complete!")


RATING ANALYSIS SUMMARY

Descriptive Statistics:
count    7105.000
mean        3.514
std         0.461
min         1.800
25%         3.200
50%         3.500
75%         3.800
max         4.900
Name: rating, dtype: float64


Skewness: -0.062
Kurtosis: -0.517


Rating Distribution:
rating_category
Average (3.0-3.5)      1976
Excellent (4.5+)         98
Good (3.5-4.0)         2803
Poor (<3.0)            1076
Very Good (4.0-4.5)    1152
Name: count, dtype: int64

✓ Rating Analysis Complete!
